# AI Model Validation: From Mathematics to Trust
## Episode 14 — Fairness, Bias, and Subgroup Validation: When Aggregate Performance Hides Unequal Failure

**Author: Vishwanath Akuthota**  
**Founder of DrPinnacle, OpenVals and The Foundry's**

Episode 13 connected technical failure to consequence. Episode 14 asks:

> **Can a model look successful overall while systematically failing some populations more than others?**

Yes. Aggregate metrics can conceal large subgroup differences.

But fairness is not simply another metric to maximize. Different mathematical fairness definitions encode different normative goals, and several cannot generally be satisfied simultaneously when base rates differ.

Central principle:

> **Fairness is not the absence of metric differences. It is evidence that model behaviour, error, access, and consequence have been examined across relevant populations in the context of intended use.**

Scientific chain:

### Population → Subgroup → Outcome → Decision → Error → Disparity → Uncertainty → Consequence → Fairness Criterion → Evidence → Governance Decision


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import platform, sys

rng=np.random.default_rng(1414)
print("Python :",sys.version.split()[0])
print("System :",platform.system(),platform.release())
print("Machine:",platform.machine())
print("NumPy  :",np.__version__)


## 1. Aggregate Performance Can Hide Unequal Failure

Suppose overall accuracy is:

### Accuracy = P(Ŷ=Y)

This does not tell us:

### P(Ŷ=Y | G=A)

or:

### P(Ŷ=Y | G=B)

where G identifies a relevant subgroup.

### Strong Aggregate Metric ≠ Uniform Subgroup Performance


## 2. Controlled Example

We construct two synthetic groups with different outcome prevalence and different score quality.

The purpose is not to represent any real demographic population. It is a mathematical laboratory for understanding subgroup metrics.


In [ ]:
nA,nB=8000,2000
g=np.array(["A"]*nA+["B"]*nB)

yA=rng.binomial(1,.45,nA)
yB=rng.binomial(1,.25,nB)
y=np.concatenate([yA,yB])

scoreA=np.clip(.18+.62*yA+rng.normal(0,.20,nA),0,1)
scoreB=np.clip(.25+.45*yB+rng.normal(0,.27,nB),0,1)
score=np.concatenate([scoreA,scoreB])
pred=(score>=.5).astype(int)

def metrics(y,p):
    tp=np.sum((y==1)&(p==1)); fn=np.sum((y==1)&(p==0))
    fp=np.sum((y==0)&(p==1)); tn=np.sum((y==0)&(p==0))
    acc=(tp+tn)/len(y)
    tpr=tp/(tp+fn) if tp+fn else np.nan
    fpr=fp/(fp+tn) if fp+tn else np.nan
    ppv=tp/(tp+fp) if tp+fp else np.nan
    sel=(tp+fp)/len(y)
    return dict(N=len(y),TP=tp,FN=fn,FP=fp,TN=tn,
                accuracy=acc,TPR=tpr,FPR=fpr,PPV=ppv,selection=sel)

print("Overall:",metrics(y,pred))
for group in ["A","B"]:
    z=g==group
    print(group,metrics(y[z],pred[z]))


## 3. Subgroup Confusion Matrices

For each group g, preserve:

### TP_g, FN_g, FP_g, TN_g

Every threshold-dependent fairness metric is ultimately related to how these counts differ across groups.

A single overall confusion matrix can hide subgroup-specific error structures.


## 4. Group-Conditioned Accuracy

### Accuracy_g = P(Ŷ=Y | G=g)

Accuracy parity would require approximately:

### Accuracy_A ≈ Accuracy_B

But equal accuracy does not imply equal false-negative rates, false-positive rates, access, calibration, or consequences.

### Equal Accuracy ≠ Equal Error Structure


## 5. Selection Rate

For group g:

### SelectionRate_g = P(Ŷ=1 | G=g)

A demographic-parity-style criterion asks for:

### P(Ŷ=1 | G=A) = P(Ŷ=1 | G=B)

This concerns decision rates, not whether the decisions are correct.


## 6. Demographic Parity Difference

For two groups:

### Δ_DP = P(Ŷ=1|A) − P(Ŷ=1|B)

A value near zero indicates similar positive-decision rates.

It does **not** establish equal opportunity, equal error rates, calibration, or equal harm.


In [ ]:
mA=metrics(y[g=="A"],pred[g=="A"])
mB=metrics(y[g=="B"],pred[g=="B"])
dp=mA["selection"]-mB["selection"]
print("Selection A:",mA["selection"])
print("Selection B:",mB["selection"])
print("Demographic parity difference A-B:",dp)


## 7. Selection-Rate Ratio

A ratio form is:

### SR_ratio = P(Ŷ=1|A) / P(Ŷ=1|B)

Ratios and differences answer different questions and can behave differently when rates are small.

A numerical ratio should not be treated as a universal legal or ethical verdict without domain-specific interpretation.


## 8. Equal Opportunity

True-positive rate for group g:

### TPR_g = P(Ŷ=1 | Y=1,G=g)

An equal-opportunity criterion requires:

### TPR_A = TPR_B

This focuses on people/cases that truly belong to the positive outcome class.

Whether this is the relevant fairness criterion depends on the decision context.


## 9. Equalized Odds

Equalized odds requires both:

### P(Ŷ=1 | Y=1,G=A) = P(Ŷ=1 | Y=1,G=B)

and:

### P(Ŷ=1 | Y=0,G=A) = P(Ŷ=1 | Y=0,G=B)

That is:

### TPR_A = TPR_B

and:

### FPR_A = FPR_B

It aligns error rates conditional on the true outcome.


In [ ]:
print("TPR gap A-B:",mA["TPR"]-mB["TPR"])
print("FPR gap A-B:",mA["FPR"]-mB["FPR"])


## 10. Predictive Parity

Positive predictive value:

### PPV_g = P(Y=1 | Ŷ=1,G=g)

Predictive parity asks:

### PPV_A = PPV_B

It concerns the meaning of a positive prediction across groups.

This is different from equal opportunity, which conditions on the true positive class rather than predicted positives.


In [ ]:
print("PPV A:",mA["PPV"])
print("PPV B:",mB["PPV"])
print("PPV gap:",mA["PPV"]-mB["PPV"])


## 11. Base Rates Matter

For group g, prevalence is:

### π_g = P(Y=1 | G=g)

Using Bayes' rule:

### PPV_g = TPR_g π_g / [TPR_g π_g + FPR_g(1−π_g)]

Therefore, even if two groups have equal TPR and FPR, different prevalence can produce different PPV.

This is a mathematical constraint, not merely an implementation bug.


In [ ]:
tpr=.8
fpr=.1
for prevalence in [.05,.10,.25,.50,.75]:
    ppv=tpr*prevalence/(tpr*prevalence+fpr*(1-prevalence))
    print(f"Prevalence={prevalence:.0%} -> PPV={ppv:.3f}")


## 12. Fairness Criteria Can Conflict

When base rates differ and prediction is imperfect, several desirable fairness conditions cannot generally all hold simultaneously.

For example, equalized error rates and predictive parity can conflict.

### Fairness Metric Selection Is a Normative Decision Informed by Mathematics

Mathematics can expose trade-offs.

It cannot decide society's values by itself.


## 13. Fairness Is Not One Number

Possible questions include:

### Are positive-decision rates similar?
### Are qualified/positive cases recognized equally?
### Are false alarms similar?
### Does a positive prediction mean the same thing?
### Are probabilities calibrated?
### Are harms distributed acceptably?
### Is access equitable?
### Are errors concentrated intersectionally?

Different questions produce different metrics.


## 14. Thresholds Change Fairness Metrics

For score s(x) and threshold τ:

### Ŷ_τ = 1[s(x)≥τ]

Then subgroup TPR, FPR, PPV and selection rate are all functions of τ.

### Fairness Metric = Function of Decision Policy

A model cannot be separated from the threshold policy used in deployment.


In [ ]:
thresholds=np.linspace(.1,.9,81)
tpr_gap=[]; fpr_gap=[]; sel_gap=[]
for t in thresholds:
    p=(score>=t).astype(int)
    a=metrics(y[g=="A"],p[g=="A"])
    b=metrics(y[g=="B"],p[g=="B"])
    tpr_gap.append(a["TPR"]-b["TPR"])
    fpr_gap.append(a["FPR"]-b["FPR"])
    sel_gap.append(a["selection"]-b["selection"])

plt.figure(figsize=(9,5))
plt.plot(thresholds,tpr_gap,label="TPR gap")
plt.plot(thresholds,fpr_gap,label="FPR gap")
plt.plot(thresholds,sel_gap,label="Selection-rate gap")
plt.axhline(0,linewidth=1)
plt.xlabel("Threshold")
plt.ylabel("Group A − Group B")
plt.title("Fairness Metrics Change With Threshold")
plt.legend()
plt.grid(alpha=.25)
plt.show()


## 15. Group-Specific Thresholds

One possible intervention is to use different thresholds by group.

This can reduce one disparity while increasing another.

It can also raise legal, ethical, governance, operational, or policy questions depending on jurisdiction and use case.

### Mathematical Feasibility ≠ Automatic Acceptability


## 16. Calibration Across Groups

For group g, ideal calibration requires:

### P(Y=1 | p̂=p,G=g) = p

If both groups receive predictions around 0.7, approximately 70% should experience the positive outcome in each group under ideal groupwise calibration.

Calibration concerns probability meaning, not equal decision rates.


## 17. Calibration and Error-Rate Parity Can Conflict

When groups have different base rates, a calibrated imperfect score generally cannot also satisfy every error-rate parity criterion simultaneously.

This is why fairness validation must state which property is being targeted and why.

### No Universal Fairness Metric Solves Every Context


## 18. Reliability Diagrams by Group

Calibration should be examined separately where subgroup sample sizes support it.

Sparse bins can create noisy apparent disparities.

The uncertainty of subgroup calibration is part of the result.


In [ ]:
def calibration_points(y,p,bins=8):
    edges=np.linspace(0,1,bins+1)
    xs=[]; ys=[]; ns=[]
    for i in range(bins):
        z=(p>=edges[i]) & (p<(edges[i+1] if i<bins-1 else edges[i+1]+1e-12))
        if z.sum():
            xs.append(p[z].mean()); ys.append(y[z].mean()); ns.append(z.sum())
    return np.array(xs),np.array(ys),np.array(ns)

plt.figure(figsize=(7,6))
for group in ["A","B"]:
    z=g==group
    xcal,ycal,ncal=calibration_points(y[z],score[z])
    plt.plot(xcal,ycal,marker="o",label=f"Group {group}")
plt.plot([0,1],[0,1],linestyle="--",label="Ideal")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed outcome frequency")
plt.title("Groupwise Calibration")
plt.legend()
plt.grid(alpha=.25)
plt.show()


## 19. Subgroup Sample Size

If subgroup g has n_g observations, metric uncertainty depends on n_g.

For a binomial proportion p̂_g:

### SE(p̂_g) ≈ √[p̂_g(1−p̂_g)/n_g]

Small groups can have large uncertainty.

### Small Observed Gap ≠ Proven Equality
### Large Observed Gap ≠ Precisely Estimated Gap


## 20. Confidence Interval for a Difference

For independent subgroup proportions p̂_A and p̂_B:

### Δ̂ = p̂_A − p̂_B

Approximate standard error:

### SE(Δ̂) = √[p̂_A(1−p̂_A)/n_A + p̂_B(1−p̂_B)/n_B]

A disparity estimate should be accompanied by uncertainty when assumptions support the approximation.


In [ ]:
def diff_interval(p1,n1,p2,n2,z=1.96):
    d=p1-p2
    se=np.sqrt(p1*(1-p1)/n1+p2*(1-p2)/n2)
    return d,d-z*se,d+z*se

d,lo,hi=diff_interval(mA["selection"],nA,mB["selection"],nB)
print("Selection-rate difference:",d)
print("Approx. 95% interval:",(lo,hi))


## 21. Rare Subgroups and Evidence Scarcity

A production population can contain small but important groups.

If n_g is tiny, “no observed failures” provides weak assurance.

Episode 12's rare-event logic still applies conditionally:

### p_fail,g,upper ≈ 3/n_g

when zero failures are observed under simplified independent conditions.


## 22. Intersectionality

Checking one attribute at a time can hide concentrated failure at intersections.

If A and B are attributes:

### P(F=1 | A=a,B=b)

may be much larger than either marginal:

### P(F=1 | A=a)

or:

### P(F=1 | B=b)

### Marginal Fairness ≠ Intersectional Fairness


In [ ]:
# Synthetic illustration: two binary attributes.
N=20000
a=rng.integers(0,2,N)
b=rng.integers(0,2,N)
p_fail=np.full(N,.02)
p_fail[(a==1)&(b==1)]=.12
fail=rng.binomial(1,p_fail)

print("Overall:",fail.mean())
for av in [0,1]:
    print("A=",av,"failure:",fail[a==av].mean())
for bv in [0,1]:
    print("B=",bv,"failure:",fail[b==bv].mean())
for av in [0,1]:
    for bv in [0,1]:
        z=(a==av)&(b==bv)
        print(f"A={av}, B={bv}: N={z.sum()}, failure={fail[z].mean():.3f}")


## 23. The Intersection Explosion

With many attributes and categories, the number of intersections can grow rapidly.

If attribute j has c_j categories:

### Number of possible intersections = Π_j c_j

This creates a statistical challenge:

### More Granularity → Better Failure Localization
### More Granularity → Less Data Per Cell

Fairness validation requires a strategy for both coverage and uncertainty.


## 24. Multiple Comparisons

If hundreds of subgroup metrics are tested, some large differences will appear by chance.

Therefore large-scale subgroup discovery may require:

- multiplicity awareness
- held-out confirmation
- hierarchical modeling
- shrinkage
- replication
- practical-significance criteria

### Searching More Groups Increases False Discovery Opportunity


## 25. Statistical Significance Is Not Fairness

A disparity can be statistically significant but operationally tiny.

A disparity can be practically severe but statistically uncertain because the subgroup is small.

### Statistical Significance ≠ Ethical Significance
### Statistical Non-Significance ≠ Evidence of Fairness


## 26. Practical Consequence

Suppose two groups differ in false-negative rate by 2 percentage points.

Whether that matters depends on:

- decision stakes
- number exposed
- severity of missed positives
- alternatives
- reversibility
- appeal mechanisms
- historical context
- legal and governance requirements

Metrics require consequence context.


## 27. Bias Has Multiple Sources

Observed disparity can arise from:

### Historical/Societal Process
### Sampling
### Measurement
### Label Definition
### Feature Construction
### Missing Data
### Model Fitting
### Threshold Policy
### Deployment
### Human Interaction
### Feedback Loops

### Model Bias ≠ Only Algorithmic Bias


## 28. Sampling Bias

If evaluation data do not represent deployment:

### P_eval(X,G,Y) ≠ P_deploy(X,G,Y)

then subgroup estimates may not generalize.

A fairness report is only as credible as its sampling design.


## 29. Measurement Bias

A variable may measure different constructs or have different error properties across groups.

If:

### X_observed = X_true + Measurement Error

and the measurement-error distribution differs by group, model disparities can emerge even before model fitting.


## 30. Label Bias

Observed label Y may be an imperfect proxy for the construct of interest Y*.

### Y ≠ Y*

If label error depends on group:

### P(Y≠Y* | G=A) ≠ P(Y≠Y* | G=B)

then apparently “objective” performance metrics can encode unequal measurement quality.


## 31. Missingness Can Be Unequal

If missing-data probability differs:

### P(Missing | G=A) ≠ P(Missing | G=B)

then imputation or fallback behaviour can create subgroup-specific error.

Fairness validation should include data-quality metrics by group.


## 32. Proxy Variables

Removing a protected or sensitive attribute does not guarantee that the model cannot infer related information from correlated variables.

### Attribute Removed ≠ Information Removed

This is why fairness cannot be established merely by excluding one column.


## 33. Fairness Through Unawareness Is Insufficient

A model can be group-blind in its explicit inputs while producing systematically different outcomes because other features encode correlated structure.

Evaluation must examine outcomes and errors, not only the feature list.


## 34. Individual and Group Fairness

Group fairness compares statistical properties across groups.

Individual fairness is often summarized conceptually as:

> Similar individuals should be treated similarly.

A mathematical form can use a task-relevant distance d:

### d_Y(f(x_i),f(x_j)) ≤ L·d_X(x_i,x_j)

But defining who is “similar” is itself a substantive modeling and governance decision.


## 35. Counterfactual Reasoning

A counterfactual fairness question asks whether a decision would change under a hypothetical change to a sensitive attribute while holding appropriate causal factors fixed.

This requires a causal model.

### Correlation-Based Fairness Metric ≠ Causal Fairness Evidence

Counterfactual claims should not be made without defensible causal assumptions.


## 36. Causal Graph Perspective

A sensitive attribute can influence:

- legitimate mediators
- illegitimate pathways
- labels
- measurements
- selection into the dataset

Determining which causal paths are acceptable is partly a normative and domain question.

Mathematics makes assumptions explicit; it does not decide them automatically.


## 37. Distribution Shift and Fairness

From Episode 11:

### P_t(X,Y,G) can change over time

Therefore:

### Fairness_t₀ ≠ Fairness_t₁

even when the model is unchanged.

Subgroup prevalence, calibration and error rates can drift independently.


## 38. Feedback Loops

Model decisions can change future data.

Example structure:

### Model Decision_t → Opportunity/Exposure_t → Observed Outcome_(t+1) → Training Data_(t+1)

This can reinforce disparities or obscure unobserved counterfactual outcomes.

### Deployment Is Part of the Data-Generating Process


## 39. Selective Labels

Sometimes outcomes are observed only for people/cases receiving a particular decision.

Then:

### Y observed only when D=1

This makes evaluation of rejected or untreated cases difficult.

Naive fairness estimates can therefore be biased by the historical decision policy.


## 40. Fairness Under Abstention

Suppose a model can abstain.

Then compare:

### Coverage_g = P(Model Answers | G=g)

and:

### SelectiveRisk_g = E[L | Model Answers,G=g]

A system can appear fair among answered cases while disproportionately abstaining on one group.

### Equal Error Among Answers ≠ Equal Access to Answers


## 41. Fairness and Harm

Metric disparity is not identical to harm disparity.

A complete analysis asks:

### Who receives errors?
### What type of errors?
### How often?
### With what consequence?
### Can the decision be appealed or corrected?
### Who bears the residual risk?

Episode 13's consequence framework therefore applies directly to fairness.


## 42. Fairness Constraints

A model-selection problem might be written:

### minimize_θ L(θ)

subject to:

### |TPR_A−TPR_B| ≤ ε

and:

### |FPR_A−FPR_B| ≤ δ

This is mathematically possible, but ε and δ are policy choices, not universal constants.


## 43. Pareto Trade-Offs

When objectives conflict, there may be no single solution that improves every metric.

A policy is Pareto-dominated if another policy improves at least one objective without worsening others.

Fairness-aware model selection can therefore examine a frontier across:

### Utility
### TPR Gap
### FPR Gap
### Calibration
### Cost
### Safety

### One Optimum ≠ Universal Optimum


## 44. Fairness Metric Gaming

Optimizing only the reported fairness metric can produce superficial compliance.

Examples include:

- shifting disparities to an unmeasured subgroup
- equalizing rates while worsening everyone
- changing thresholds without addressing data quality
- improving averages while increasing severe tail harm

### Metric Compliance ≠ Substantive Fairness


## 45. Minimum Reporting Set

A defensible subgroup validation report should include:

- intended use
- group definitions and rationale
- sample sizes
- prevalence
- confusion matrices
- accuracy
- TPR/recall
- FPR
- specificity
- precision/PPV
- selection rate
- calibration
- uncertainty intervals
- intersectional results
- missingness/data quality
- threshold policy
- consequences of FP/FN
- temporal validity
- known limitations

The exact set should be adapted to the application.


## 46. Do Not Rank People by a Fairness Score

A fairness metric evaluates a system or decision process under a defined criterion.

It should not become a mechanism for assigning moral worth or intrinsic value to individuals or groups.

### System Evaluation ≠ Human Valuation


## 47. OpenVals Perspective

A validation platform should preserve:

- intended use and decision context
- subgroup definitions
- rationale for each fairness criterion
- subgroup sample size and prevalence
- subgroup confusion matrices
- selection rates
- TPR/FPR/FNR/TNR
- predictive values
- calibration
- metric gaps and ratios
- uncertainty
- intersectional coverage
- missingness and label-quality evidence
- threshold policy
- fairness constraints
- consequences by error type
- drift over time
- feedback-loop risks
- unresolved evidence gaps
- accountable review decision

The purpose is not to produce a universal **Fairness Score**.

It is to make subgroup evidence inspectable, contextual and challengeable.


## 48. Scientific Fairness Chain

### Population → Subgroup → Outcome → Decision → Error → Disparity → Uncertainty → Consequence → Fairness Criterion → Evidence → Governance Decision → Monitoring

Fairness validation becomes an evidence process rather than a checkbox.


## 49. Mathematical Summary

### Accuracy_g=P(Ŷ=Y|G=g)

### SelectionRate_g=P(Ŷ=1|G=g)

### Δ_DP=P(Ŷ=1|A)−P(Ŷ=1|B)

### TPR_g=P(Ŷ=1|Y=1,G=g)

### FPR_g=P(Ŷ=1|Y=0,G=g)

### PPV_g=P(Y=1|Ŷ=1,G=g)

### PPV_g = TPR_gπ_g / [TPR_gπ_g + FPR_g(1−π_g)]

### Equal Opportunity: TPR_A=TPR_B

### Equalized Odds: TPR_A=TPR_B and FPR_A=FPR_B

### Predictive Parity: PPV_A=PPV_B

### Groupwise Calibration: P(Y=1|p̂=p,G=g)=p

### SE(p̂_g)≈√[p̂_g(1−p̂_g)/n_g]

### Δ̂=p̂_A−p̂_B

### SE(Δ̂)=√[p̂_A(1−p̂_A)/n_A + p̂_B(1−p̂_B)/n_B]

### p_fail,g,upper≈3/n_g  for zero observed failures under simplified assumptions

### Number of intersections=Π_j c_j

### Coverage_g=P(Model Answers|G=g)

### SelectiveRisk_g=E[L|Model Answers,G=g]


## 50. Central Conclusion

### Aggregate Performance ≠ Subgroup Fairness
### Equal Accuracy ≠ Equal Error Structure
### Metric Parity ≠ Equal Harm
### Statistical Significance ≠ Ethical Significance
### Attribute Removal ≠ Bias Removal
### One Fairness Metric ≠ Universal Fairness

> **Fairness cannot be validated by asking whether one number is equal. It requires asking which outcomes, errors, opportunities, and consequences should be comparable—and why.**

The scientific questions are:

### Which populations can be affected?
### Are they represented in the evidence?
### Where do errors concentrate?
### Are differences larger than sampling uncertainty?
### Which fairness criterion matches the decision context?
### Which criteria mathematically conflict?
### What consequences do disparities create?
### Are intersectional groups hidden by aggregate analysis?
### Can labels or measurements themselves be biased?
### Does fairness persist after deployment?

This gives OpenVals another core principle:

> **Fairness is not a universal score. It is a traceable set of claims, assumptions, measurements, uncertainties, trade-offs, and consequences that must remain visible throughout the AI lifecycle.**

## Next Episode

**Episode 15 — Explainability and Interpretability: Can We Understand Why the Model Made This Decision?**

Episode 15 can distinguish interpretability from post-hoc explanation, develop local and global explanations, feature attribution, permutation importance, partial dependence, Shapley-value foundations, counterfactual explanations, explanation stability, faithfulness, causal limitations, and why a plausible explanation is not automatically a true explanation.
